#### Regularization: Penalized regression

The most common penalized regression models are:

1. Ridge regression
2. Lasso regression
3. Elastic Net regression

Here, we will explore the three methods and compare their results with a **Multiple Linear regression** model.

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Load the Data set
cars_data =  pd.read_csv("Span_new.csv")
cars_data.head()

,Unnamed: 0,ID,make,model,months_old,power,gear_type,fuel_type,kms,price,age
0,0,97860,Porsche,911,240,210,manual,gasoline,202000,999999,1998
1,1,27821,Ford,Mustang,54,487,manual,gasoline,30000,685000,2013
2,2,97801,Porsche,911,358,220,manual,gasoline,56300,555555,1988
3,3,98251,Porsche,911,14,368,manual,gasoline,2800,470000,2016
4,4,98250,Porsche,911,3,515,unknown,gasoline,10,450000,2017


##### Pre-Processing

In [4]:
print(cars_data.duplicated().values.any(),"\n")
print(cars_data.isnull().values.any())

False 

False


In [5]:
#  Deleting the unwanted columns (Unnamed: 0 & ID) & Changing the name of the variable from age to model_year
cars_data = cars_data.drop(['Unnamed: 0', 'ID'], axis = 1)
cars_data = cars_data.rename({'age' : 'model_year'}, axis = 1)
print(cars_data.shape)
cars_data.head()

(197, 9)


,make,model,months_old,power,gear_type,fuel_type,kms,price,model_year
0,Porsche,911,240,210,manual,gasoline,202000,999999,1998
1,Ford,Mustang,54,487,manual,gasoline,30000,685000,2013
2,Porsche,911,358,220,manual,gasoline,56300,555555,1988
3,Porsche,911,14,368,manual,gasoline,2800,470000,2016
4,Porsche,911,3,515,unknown,gasoline,10,450000,2017


In [6]:
variables_in_study = cars_data[['months_old', 'power','kms','price']]

# standardizing features
scaler = StandardScaler() 
scaler.fit(variables_in_study)
variables_in_study = scaler.transform(variables_in_study) 

variables_in_study = pd.DataFrame(variables_in_study, columns = ['months_old', 'power','kms','price'])
independent_variables = variables_in_study[['months_old', 'power','kms']]
dependent_variable = variables_in_study['price']

In [7]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(independent_variables, dependent_variable, test_size=0.2, random_state=0)

#### A) Training the models 

We will train 4 different models:
1. Linear regression (model_linear)
2. Ridge regression (model_ridge)
3. Lasso regression (model_lasso)
4. Elastic Net regression (model_net)

In [8]:
# Call the Models
models = {
    "Multiple Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1),
    "Lasso Regression": Lasso(alpha=0.03),
    "Elastic Net Regression": ElasticNet(alpha=0.01, l1_ratio=0.8)
}

In [9]:
# Store the Models Results
results = []

# Train, Prediction & Evaluate Models
for model_name, model in models.items():
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)
    
    # Training Metrics 
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    
    # Testing Metrics 
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    # Append results
    results.append({
        "Model": model_name,
        "Train MSE": train_mse,
        "Test MSE": test_mse,
        "Train MAE": train_mae,
        "Test MAE": test_mae,
    })

In [10]:
# Convert the results into Pandas Table
results_df = pd.DataFrame(results)

# Display Results
print("\nModel Performance Comparison Table:\n")
print(results_df)


Model Performance Comparison Table:

                        Model  Train MSE  Test MSE  Train MAE  Test MAE
0  Multiple Linear Regression   0.848331  0.864838   0.538025  0.588982
1            Ridge Regression   0.848363  0.860760   0.536793  0.587804
2            Lasso Regression   0.853459  0.814514   0.524407  0.577339
3      Elastic Net Regression   0.848763  0.849248   0.533490  0.584300


#### Cross Validation 

Let's use cross validation to find the optimal Lambdas (alphas) for different models. 

In [12]:
# Find Best Alpha Values using Cross Validation

# Alpha values to test
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]

# Ridge CV
ridge_cv = RidgeCV(alphas=alpha_values, cv=5)
ridge_cv.fit(X_train, y_train)
best_alpha_ridge = ridge_cv.alpha_

# Lasso CV
lasso_cv = LassoCV(alphas=alpha_values, cv=5)
lasso_cv.fit(X_train, y_train)
best_alpha_lasso = lasso_cv.alpha_

# Elastic Net CV
elastic_cv = ElasticNetCV(
    alphas=alpha_values,
    l1_ratio=[0.2, 0.5, 0.8],
    cv=5
)
elastic_cv.fit(X_train, y_train)

best_alpha_elastic = elastic_cv.alpha_
best_l1_ratio = elastic_cv.l1_ratio_

# Print best values
print("\nBest Hyperparameters \n")
print("Best Ridge Alpha:", best_alpha_ridge)
print("Best Lasso Alpha:", best_alpha_lasso)
print("Best ElasticNet Alpha:", best_alpha_elastic)
print("Best ElasticNet l1_ratio:", best_l1_ratio)


Best Hyperparameters 

Best Ridge Alpha: 100.0
Best Lasso Alpha: 100.0
Best ElasticNet Alpha: 100.0
Best ElasticNet l1_ratio: 0.2


In [13]:
# Define Final Models with Best Hyperparameters
models = {
    "Multiple Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=best_alpha_ridge),
    "Lasso Regression": Lasso(alpha=best_alpha_lasso),
    "Elastic Net Regression": ElasticNet(
        alpha=best_alpha_elastic,
        l1_ratio=best_l1_ratio
    )
}

In [14]:
# Store the Models Results
results = []

# Train, Prediction & Evaluate Models
for model_name, model in models.items():

    # Train model
    model.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    # Training Metrics
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)

    # Testing Metrics
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    # Save results
    results.append({
        "Model": model_name,
        "Train MSE": train_mse,
        "Test MSE": test_mse,
        "Train MAE": train_mae,
        "Test MAE": test_mae,
    })

In [15]:
# Create Performance Comparison Table
results_df = pd.DataFrame(results)

# Round for better display
results_df = results_df.round(4)

print("\nModel Performance Comparison\n")
print(results_df)


Model Performance Comparison

                        Model  Train MSE  Test MSE  Train MAE  Test MAE
0  Multiple Linear Regression     0.8483    0.8648     0.5380    0.5890
1            Ridge Regression     0.9081    0.7231     0.5158    0.5739
2            Lasso Regression     1.0903    0.6457     0.5121    0.5510
3      Elastic Net Regression     1.0903    0.6457     0.5121    0.5510
